<a href="https://colab.research.google.com/github/carlavilla/GIS/blob/main/PS4_GIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Environment Set Up

In [1]:
!pip install mapclassify

In [2]:
import os, zipfile
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import geopandas as gpd

import mapclassify

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from google.colab import files

from google.colab.data_table import DataTable
DataTable.max_columns = 250

#Dataset 1: UFO Sightings in the US
UFO sighting reports to the National UFO Reporting Center for the year 2024. </br>

Source: [NUFORC Databank](https://nuforc.org/databank/) </br>
Data was sorted for USA sighting reports on the website and recorded onto an excel spreadsheet for this project. Since my interest is in attitudes towards UFO sightings in a given year, this dataset contains reports made in 2024, regardless of the actual supposed sighting of the object. That means that some of the sightings are from previous years and they were only reported in 2024; conversely, some of the sightings claimed to have occurred in 2024 are not included in this dataset and they were reported in 2025. Other interesting variables include date of sighting, city of report, shape of object, and alternative explanation for event. Although some events appear to be disproven, all reports are used as indicative of supernatural beliefs. </br>

##UFO Sighting Reports dataset comments:
Since dataset includes single entries of reports, the count for the number of reports in each state in 2024 was calculated. This count appears in python as a series, but this series will be used as the variable to merge with the map. </br>

In [3]:
! wget -q -O ufo2024.xlsx https://raw.githubusercontent.com/carlavilla/GIS/main/ufo2024.xlsx
ufo=pd.read_excel('ufo2024.xlsx', engine='openpyxl')

In [4]:
ufo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4034 entries, 0 to 4033
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Occurred     4034 non-null   datetime64[ns]
 1   City         3974 non-null   object        
 2   State        4034 non-null   object        
 3   Country      4034 non-null   object        
 4   Shape        4032 non-null   object        
 5   Summary      4034 non-null   object        
 6   Reported     4034 non-null   datetime64[ns]
 7   Media        1589 non-null   object        
 8   Explanation  1007 non-null   object        
dtypes: datetime64[ns](2), object(7)
memory usage: 283.8+ KB


In [5]:
ufo.rename(columns={'State': 'state'}, inplace=True)

In [6]:
#getting the count by state
from collections import Counter
ufocount = ufo['state'].value_counts()
print(ufocount)

state
CA    418
FL    244
TX    228
NY    221
WA    190
AZ    175
PA    157
CO    137
NC    125
MI    117
OH    113
NJ    101
OR     98
MA     96
GA     91
VA     91
IL     87
MN     83
TN     75
IN     69
SC     62
WI     61
KY     60
UT     60
MD     60
MO     59
CT     54
ID     54
OK     51
AL     50
NV     50
LA     47
NM     46
ME     45
KS     39
MT     37
AR     35
IA     34
WV     31
MS     28
NH     25
SD     20
AK     18
VT     18
NE     18
DE     17
RI     15
HI     11
WY      6
DC      5
ND      2
Name: count, dtype: int64


In [7]:
#getting the count by city
from collections import Counter
ufocity = ufo['City'].value_counts()
print(ufocity)

City
New York        48
Los Angeles     40
San Diego       24
Las Vegas       20
Spokane         19
                ..
Willow           1
Fairbanks        1
Dillingham       1
Palmer           1
Anchor Point     1
Name: count, Length: 2369, dtype: int64


In [9]:
ufo.head(5)
ufo.columns

,Occurred,City,state,Country,Shape,Summary,Reported,Media,Explanation
0,2024-12-15 17:00:00,Chatanika,AK,USA,Formation,Saw 3 non moving lights that disappeared one a...,2024-12-31,NaN,NaN
1,2024-12-28 08:01:00,Meadow Lakes,AK,USA,Circle,Observed flashing light (after larger plane we...,2024-12-28,Y,Drone
2,2024-12-13 20:00:00,NaN,AK,USA,Chevron,Large V shaped triangle with white lights,2024-12-25,NaN,NaN
3,2024-12-19 23:30:00,Trapper Creek,AK,USA,Oval,Green glowing oval object the size of a school...,2024-12-24,Y,NaN
4,2024-11-02 23:30:00,Fox,AK,USA,Orb,Floating light that accelerated downwards into...,2024-12-22,NaN,NaN


Index(['Occurred', 'City', 'state', 'Country', 'Shape', 'Summary', 'Reported',
       'Media', 'Explanation'],
      dtype='object')

In [10]:
ufo_co = ufo[ufo['state'] == 'CO']
display(ufo_co.head())

,Occurred,City,state,Country,Shape,Summary,Reported,Media,Explanation
696,2024-06-21 20:58:00,Thornton,CO,USA,Orb,We had witnessed at least six orbs being stati...,2024-12-31,Y,NaN
697,2024-08-21 22:50:00,Colorado Springs,CO,USA,Orb,I sent picture and videos to TV station KRDO. ...,2024-12-30,Y,Drone?
698,2023-07-28 20:00:00,Ridgway,CO,USA,Unknown,No moving Massive Lights in arch shape,2024-12-28,Y,NaN
699,2024-07-20 18:00:00,Denver,CO,USA,Cube,Very noticeable arc bright light very high in ...,2024-12-27,Y,NaN
700,2024-12-20 04:00:00,Lakewood,CO,USA,Sphere,It was an orb that was moving closer and farth...,2024-12-21,NaN,NaN


In [13]:
from geopy.geocoders import Nominatim
#if you do a lot of geocoding: https://developers.google.com/maps/documentation/geocoding/overview

#change email addr!! they will blcok you!
geolocator = Nominatim(user_agent='tutu@gmail.com')
loc = geolocator.geocode("401 cooper st, camden nj 08102")
loc.address
loc[1]

'401, Cooper Street, Downtown, Camden, Camden County, New Jersey, 08102, United States of America'

(39.9472224, -75.1220198)

In [19]:
import folium as f
from folium.plugins import MarkerCluster, HeatMap

import time

In [14]:
ufo_co['combined'] = ufo_co[['City','state']].apply(lambda row: ' '.join(row.values.astype(str)), axis=1)
ufo_co.head()

/tmp/ipython-input-1471205263.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ufo_co['combined'] = ufo_co[['City','state']].apply(lambda row: ' '.join(row.values.astype(str)), axis=1)


,Occurred,City,state,Country,Shape,Summary,Reported,Media,Explanation,combined
696,2024-06-21 20:58:00,Thornton,CO,USA,Orb,We had witnessed at least six orbs being stati...,2024-12-31,Y,NaN,Thornton CO
697,2024-08-21 22:50:00,Colorado Springs,CO,USA,Orb,I sent picture and videos to TV station KRDO. ...,2024-12-30,Y,Drone?,Colorado Springs CO
698,2023-07-28 20:00:00,Ridgway,CO,USA,Unknown,No moving Massive Lights in arch shape,2024-12-28,Y,NaN,Ridgway CO
699,2024-07-20 18:00:00,Denver,CO,USA,Cube,Very noticeable arc bright light very high in ...,2024-12-27,Y,NaN,Denver CO
700,2024-12-20 04:00:00,Lakewood,CO,USA,Sphere,It was an orb that was moving closer and farth...,2024-12-21,NaN,NaN,Lakewood CO


In [23]:
ufo_co_test = ufo_co.head(10)
display(ufo_co_test)

,Occurred,City,state,Country,Shape,Summary,Reported,Media,Explanation,combined
696,2024-06-21 20:58:00,Thornton,CO,USA,Orb,We had witnessed at least six orbs being stati...,2024-12-31,Y,NaN,Thornton CO
697,2024-08-21 22:50:00,Colorado Springs,CO,USA,Orb,I sent picture and videos to TV station KRDO. ...,2024-12-30,Y,Drone?,Colorado Springs CO
698,2023-07-28 20:00:00,Ridgway,CO,USA,Unknown,No moving Massive Lights in arch shape,2024-12-28,Y,NaN,Ridgway CO
699,2024-07-20 18:00:00,Denver,CO,USA,Cube,Very noticeable arc bright light very high in ...,2024-12-27,Y,NaN,Denver CO
700,2024-12-20 04:00:00,Lakewood,CO,USA,Sphere,It was an orb that was moving closer and farth...,2024-12-21,NaN,NaN,Lakewood CO
701,2024-12-14 18:37:00,Denver,CO,USA,Orb,Multiple Orbs Following Same Path,2024-12-20,Y,NaN,Denver CO
702,2024-07-21 22:35:00,Grand Junction,CO,USA,Triangle,Black triangle with three flashing white light...,2024-12-19,Y,Drone?,Grand Junction CO
703,2024-12-14 20:30:00,Commerce City,CO,USA,Circle,Dozens of lights heading towards northern DIA ...,2024-12-18,NaN,NaN,Commerce City CO
704,2024-12-18 03:12:00,Aurora,CO,USA,Orb,Was moving like a airplane at first then stopp...,2024-12-18,NaN,Drone?,Aurora CO
705,2024-12-16 17:38:00,Elbert,CO,USA,Light,While driving on Walker road looking north spo...,2024-12-16,Y,Aircraft?,Elbert CO


In [24]:
import time
from geopy.geocoders import Nominatim

# Re-initialize geolocator with a longer timeout
geolocator = Nominatim(user_agent='tutu@gmail.com', timeout=10)

m = f.Map(location=geolocator.geocode("colorado")[1], zoom_start=9)
for i in ufo_co_test.index:
  time.sleep(3) # Keep sleep time to mitigate rate limiting issues
  try:
    location = geolocator.geocode(ufo_co_test['combined'][i])
    if location:
      f.Marker(location[1], popup='Shape: '+ ufo_co_test['Shape'][i]).add_to(m)
    else:
      print(f"Could not geocode: {ufo_co_test['combined'][i]}")
  except Exception as e:
    print(f"Error geocoding {ufo_co_test['combined'][i]}: {e}")
#m.save('m7.html')
m